# Machine-Learning ADC Target Ranking

**v0.1 educational notebook**

## Goal
Compare transparent rule-based ranking with data-driven target-likeness models. The example labels are deliberately small and educational; unlabeled genes are **not** true negatives.

## Canonical guide

### Learning objectives
- Train simple interpretable and nonlinear target-likeness models.
- Use ranking-oriented metrics under severe class imbalance.
- Compare ML ranking with the rule-based baseline.

### Data mode
Synthetic/public-data-style target features plus a deliberately small educational positive-label list.

### Inputs
`outputs/adc_target_feature_matrix_v01.csv`, `outputs/adc_rule_based_ranking_v01.csv`, `data/example/known_adc_targets_v01.csv`

### Canonical outputs
`outputs/adc_ml_cv_predictions_v01.csv`, `outputs/adc_ml_ranking_v01.csv`, `outputs/adc_ml_model_summary_v01.csv`, `outputs/adc_rule_ml_combined_v01.csv`

### Scientific limitation
Unlabeled genes are not true negatives. With only a few example positives, performance estimates are unstable and should be read as workflow demonstrations.

### Next step
Notebook 07 reframes the problem explicitly as Positive–Unlabeled learning.


In [1]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'
OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)
print('ROOT:', ROOT.resolve())

ROOT: /mnt/data/audit_adc/adc-scientific-ai


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier
f=pd.read_csv(OUTPUTS/'adc_target_feature_matrix_v01.csv',index_col=0)
labels=pd.read_csv(DATA/'example'/'known_adc_targets_v01.csv')
known=set(labels.gene_symbol)
cols=['tumor_score','selectivity_score','prevalence_score','surface_score_norm','heterogeneity_risk','normal_rna_risk','normal_protein_risk_norm','critical_organ_risk']
d=f[f.surface_score.ge(.5)].dropna(subset=cols).copy(); d['label']=d.index.isin(known).astype(int)
X=d[cols]; y=d.label
print(y.value_counts())

label
0    20
1     4
Name: count, dtype: int64


In [3]:
n_splits=min(3,int(y.sum()))
if n_splits<2: raise ValueError('Need at least two positives for CV')
cv=StratifiedKFold(n_splits=n_splits,shuffle=True,random_state=42)
log=Pipeline([('scale',StandardScaler()),('clf',LogisticRegression(class_weight='balanced',max_iter=2000,random_state=42))])
rf=RandomForestClassifier(n_estimators=120,class_weight='balanced',min_samples_leaf=2,random_state=42,n_jobs=1)
spw=(len(y)-y.sum())/y.sum()
xgb=XGBClassifier(n_estimators=80,max_depth=3,learning_rate=.04,subsample=.8,colsample_bytree=.8,scale_pos_weight=spw,eval_metric='logloss',random_state=42,n_jobs=1,tree_method='hist')
models={'Logistic':log,'Random Forest':rf,'XGBoost':xgb}
preds={}; rows=[]
for name,m in models.items():
    p=cross_val_predict(m,X,y,cv=cv,method='predict_proba')[:,1]; preds[name]=p
    rows.append((name,average_precision_score(y,p),roc_auc_score(y,p)))
summary=pd.DataFrame(rows,columns=['Model','Average Precision','ROC-AUC']); summary

,Model,Average Precision,ROC-AUC
0,Logistic,1.000000,1.000
1,Random Forest,1.000000,1.000
2,XGBoost,0.916667,0.975


In [4]:
pred_table=d.copy()
for name,p in preds.items(): pred_table[name.lower().replace(' ','_')+'_cv_score']=p
for m in models.values(): m.fit(X,y)
ensemble=pd.DataFrame(index=d.index)
ensemble['label']=y
ensemble['logistic_score']=log.predict_proba(X)[:,1]
ensemble['rf_score']=rf.predict_proba(X)[:,1]
ensemble['xgb_score']=xgb.predict_proba(X)[:,1]
ensemble['ensemble_score']=ensemble[['logistic_score','rf_score','xgb_score']].mean(axis=1)
ensemble['model_disagreement']=ensemble[['logistic_score','rf_score','xgb_score']].std(axis=1)
ensemble=ensemble.sort_values('ensemble_score',ascending=False)
rule=pd.read_csv(OUTPUTS/'adc_rule_based_ranking_v01.csv',index_col=0)
combined=ensemble.join(rule[['adc_target_score','stability_score']],how='left')
pred_table.to_csv(OUTPUTS/'adc_ml_cv_predictions_v01.csv')
ensemble.to_csv(OUTPUTS/'adc_ml_ranking_v01.csv')
summary.to_csv(OUTPUTS/'adc_ml_model_summary_v01.csv',index=False)
combined.to_csv(OUTPUTS/'adc_rule_ml_combined_v01.csv')
ensemble.head(10)

,label,logistic_score,rf_score,xgb_score,ensemble_score,model_disagreement
gene_symbol,,,,,,
NECTIN4,1,0.959346,0.910665,0.932418,0.934143,0.024386
FOLR1,1,0.945226,0.853792,0.932418,0.910479,0.049508
ERBB2,1,0.917284,0.853731,0.937616,0.902877,0.043759
TACSTD2,1,0.936763,0.808885,0.938495,0.894714,0.074335
GENE_G,0,0.317756,0.184688,0.134075,0.212173,0.094875
EPCAM,0,0.048982,0.182282,0.135674,0.122313,0.067647
GENE_A,0,0.111868,0.143997,0.099418,0.118428,0.023002
GENE_L,0,0.244212,0.000000,0.059632,0.101281,0.127322
GENE_R,0,0.042032,0.150421,0.099418,0.097291,0.054225


## Scientific caution
The labels represent “known educational positives” versus “currently unlabeled.” Performance metrics therefore measure enrichment for known positives, not definitive discovery accuracy. This limitation motivates Positive–Unlabeled learning in Notebook 07.